# Pivot xslx functions 0.1

Created: 2026-09-04

Develop functions for accessing data in xlsx workbooks that contain pivot tables.

This notebook assembles basic functionality needed to download a pivot Excel file and convert it to a pandas dataframe. Subsequent notebooks will deal with placing this and accessory functionality in a `pvtxl` module.

To be consistent with the API, this module will provide data in column-oriented orientation. (See [Data orientation](https://en.wikipedia.org/wiki/Data_orientation)). Pivot tables in the Excel tables provided by e-Stat are already in column-orientation. Other Excel tables will be read into column-orientation.

While the e-Stat data portal contains many datasets that are accessible through its API, a lot of the data is available only as xsl or xslx workbooks. The xslx workbooks are further divided between pivot workbooks and reporting workbooks. The purpose of this notebook is to develop an 'estatjp' module for pivot workbooks. 

<cite data-cite="basha2024pivot">Basha (2024)</cite> provides a guide for accessing raw data from pivot tables but the guide doesn't provide for the case when the pivot cache has not been saved with the file (See [Reduce the file size of your Excel spreadsheets](https://support.microsoft.com/en-us/office/reduce-the-file-size-of-your-excel-spreadsheets-c4f69e3a-8eea-4e9d-8ded-0ac301192bf9)). The module developed here must provide for both cases.

## Data source

Pivot workbooks of immediate interest to this developer are in the [statistics on registered foreign residents that are provided by the Immigration Services Agency](https://www.moj.go.jp/isa/policies/statistics/toukei_ichiran_touroku.html). The URLs in the code below are for tables of foreign residents by prefecture and by municipality.

In [1]:
xb1url = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040379765&fileKind=0" # 25-06-t1 - June 2025 by prefecture
xb2url = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040379766&fileKind=0" # 25-06-t2 - June 2025 by municipality
xb3url = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040228086&fileKind=0" # 24-06-t2 - June 2024 by municipality
xb4url = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040186956&fileKind=0" # 23-12-t1 - Dec 2023 by prefecture
xb5url = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040186957&fileKind=0" # 23-12-t2 - Dec 2023 by municipality
xb6url = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040228085&fileKind=0" # 24-06-t2 - June 2024 by prefecture

## Required packages

Note that downloading these files does not require an API ID.

In [9]:
%%capture capt
# %%capture puts the output in 'capt' from which it can be ignored or printed
%pip install pandas
%pip install plotly 
%pip install fsspec # to help pandas IO. See https://pandas.pydata.org/docs/user_guide/io.html#reading-writing-remote-files
%pip install requests
%pip install openpyxl
%pip install pyrfc6266 #

## download_to_temp(url)



The name of file being downloaded is in the response' Content-Disposition header. The following code borrows from 
[stackoverflow source](https://stackoverflow.com/questions/37060344/how-to-determine-the-filename-of-content-downloaded-with-http-in-python), which uses [pyrfc6266](https://pypi.org/project/pyrfc6266/) ([Github](https://github.com/JohnDoee/pyrfc6266)).

In [33]:
import requests
import pyrfc6266
from pathlib import Path

def dev_download_to_folder(
        url, 
        datapath = None):
    """
    Downloads a file from the given URL into a temporary file.
    Returns the temporary file path.
    The file will be deleted automatically when closed (delete=True).
    """
    try:
        # Stream download to avoid loading the whole file into memory
        with requests.get(url, stream=True, timeout=15) as response:
            response.raise_for_status()  # Raise HTTPError for bad responses

            # Get file name
            content_disp = response.headers.get("Content-Disposition", "Unknown")
            attachedfile = pyrfc6266.parse_filename(content_disp)
            savepath = Path(datapath / attachedfile)
            with savepath.open(mode='bw') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:  # Filter out keep-alive chunks
                        f.write(chunk)
                
                f.close()
            return savepath

    except requests.exceptions.RequestException as e:
        print(f"Download failed: {e}")
        return None

In [55]:
import os
import openpyxl
import numpy as np
import pandas as pd
from openpyxl import worksheet

ROOT_DIR = Path.cwd().parent.parent
dataraw = ROOT_DIR / Path(".data/raw")
temp_xlsx = dev_download_to_folder(xb6url, dataraw)

# Load an Excel file
wb = openpyxl.load_workbook(temp_xlsx)
snames = wb.sheetnames
print(snames)

['PVT', '令和６年６月末']


In [56]:
ws = wb['PVT']
# Identify the pivot table
pivot_name = ""
for p in ws._pivots:
    print(p.name)
    pivot_name=p.name #This assigns the name of the desired Pivot Table to the variable

ピボットテーブル2


In [57]:
#extract the pivot table object
pivot_table = [p for p in ws._pivots if p.name == pivot_name][0]
print(pivot_name)

ピボットテーブル2


In [58]:
# Extract cache field information from pivot table
fields_map = {}
column_names = []
rows = []
if pivot_table.cache.records != None:
    for field in pivot_table.cache.cacheFields:
        if field.sharedItems != None:
            if field.sharedItems.count > 0:
                l = []
                for f in field.sharedItems._fields:
                    try:
                        l += [f.v]
                    except AttributeError:
                        l += [""]
                fields_map[field.name] = l
    # Extract rows from pivot table cache into dict
    column_names = [field.name for field in pivot_table.cache.cacheFields]
    # print column names
    for field in pivot_table.cache.cacheFields:
        if field.sharedItems != None:
            if field.sharedItems.count > 0:
                print(field.name)

        for record in pivot_table.cache.records.r:
            # If some field in the record is missing, replace it by NaN
            record_values = [
                field.v if not isinstance(field, openpyxl.pivot.fields.Missing) else np.nan for field in record._fields
            ]

            row_dict = {k: v for k, v in zip(column_names, record_values)}

            # Shared fields are mapped as an Index, so we replace the field index by its value
            for key in fields_map:
                row_dict[key] = fields_map[key][row_dict[key]]
            
            rows.append(row_dict)

elif pivot_table != None:
    # even without records, the pivot_table should include the location of the original table
    range = worksheet.cell_range.CellRange(range_string=pivot_table.location.ref)
    top_row = {k: v.value for k, v in ws._cells.items() if k in range.top}
    column_names = top_row.values()
    range.shrink(top=1)
    for row in range.rows:
        row_values = {k: v.value for k, v in ws._cells.items() if k in row }
        row_dict = {k: v for k, v in zip(column_names, row_values.values())}
        rows.append(row_dict)

column_names

国籍・地域
在留資格
性別
年齢
都道府県


['国籍・地域', '在留資格', '性別', '年齢', '都道府県', '在留外国人数']

## make DataFrame

In [59]:
df = pd.DataFrame.from_dict(rows)

df

,国籍・地域,在留資格,性別,年齢,都道府県,在留外国人数
0,01_001：アフガニスタン,03：教授,1:男,34歳,04：宮城県,1.0
1,01_001：アフガニスタン,03：教授,1:男,34歳,08：茨城県,1.0
2,01_001：アフガニスタン,03：教授,1:男,34歳,45：宮崎県,1.0
3,01_001：アフガニスタン,03：教授,1:男,37歳,23：愛知県,1.0
4,01_001：アフガニスタン,03：教授,1:男,41歳,13：東京都,1.0
...,...,...,...,...,...,...
2609827,07_000：無国籍,39：特別永住者,2:女,80歳以上,08：茨城県,1.0
2609828,07_000：無国籍,39：特別永住者,2:女,80歳以上,13：東京都,1.0
2609829,07_000：無国籍,39：特別永住者,2:女,80歳以上,27：大阪府,1.0
2609830,07_000：無国籍,39：特別永住者,2:女,80歳以上,28：兵庫県,2.0
